In [ ]:
import os
import pandas as pd
import plotly.graph_objects as go
import pickle
import math
import numpy as np
import json

from random import sample

In [ ]:
data_directory = '/home/danielb/Desktop/Coding Projects/lidar_object_tracking/data/raw/'

frame_dict = {}
for dir in os.scandir(data_directory+'004/lidar/'):
    with open(dir, 'rb') as file:
        try:
            frame_dict[int(dir.name[0:2])] = pd.DataFrame(pickle.load(file))
        except:
            pass

frame_dict = dict(sorted(frame_dict.items()))

In [ ]:
import math

def rotate_45(x_val, y_val):
    angle = -math.pi/4
    x_rot = x_val*math.cos(angle) + y_val*math.sin(angle)
    y_rot = -x_val*math.sin(angle) + y_val*math.cos(angle)
    return x_rot,y_rot


for i in frame_dict.keys():
    frame_dict[i]['xr'],frame_dict[i]['yr'] = rotate_45(frame_dict[i]['x'],frame_dict[i]['y'])[0],rotate_45(frame_dict[i]['x'],frame_dict[i]['y'])[1]

for i in frame_dict.keys():
    frame_dict[i].drop(frame_dict[i][(frame_dict[i]['xr'].apply(abs)>25)|
                                     (frame_dict[i]['yr']<0)|
                                     (frame_dict[i]['yr'] >25)|
                                     (frame_dict[i]['z']>5)|
                                     (frame_dict[i]['z']<0)].index, inplace = True)

# for i in frame_dict.keys():
#    frame_dict[i]['z_zero'] = frame_dict[i]['z'].apply(lambda z: z*0)


In [ ]:
sub_sample = [sample(list(frame_dict[i].index), 1500) for i in range(len(frame_dict))]

coord_by_frame = {i:frame_dict[i][['xr','yr','z']].loc[sub_sample[i]] for i in range(len(frame_dict))}

In [ ]:
fig = go.Figure()

for i in range(80):
    fig.add_trace(
        go.Scatter3d(
            visible = False,
            mode = 'markers',
            marker = dict(size = 2),
            x = coord_by_frame[i]['xr'],
            y = coord_by_frame[i]['yr'],
            z = coord_by_frame[i]['z']
        )
    )
    fig.update_layout(height = 500, width = 750)
    fig.update_scenes(xaxis =dict(range = [-25,25]), 
                      yaxis=dict(range = [0,25]), 
                      zaxis=dict(range = [0,5]),
                      aspectmode = 'data')

fig.data[0].visible = True

steps = []
for i in range(len(fig.data)):
    step = dict(
        method = 'update',
        args= [{'visible':[False]*len(fig.data)}],
        label = f'{i/10}s',
    )
    step['args'][0]['visible'][i] = True
    steps.append(step)

sliders = [dict(
    active = 0,
    currentvalue = {'prefix': "Frame: ",},
    pad={'t':80},
    steps = steps
)]

fig.update_layout(
    sliders = sliders
)

fig.show()